# Working with Census Geographies

morpc-census provides tools to fetch Census geometries and data for specific places at specific resolutions. Every query is defined by two things:

- **Scope** — *where* (Franklin County, the 15-county MORPC region, the state of Ohio)
- **Scale** — *at what resolution* (county, census tract, place)

The results come back with a **GEOIDFQ** — a fully-qualified geographic identifier — for each geography returned.

In [ ]:
from morpc_census import SCOPES, valid_scale, fetch_geos_from_scale_scope, GeoIDFQ

## 1. Scopes — choosing where

A scope names a geographic extent. morpc-census ships with built-in scopes for the US, all 50 states, all Ohio counties, and several MORPC regions.

In [ ]:
# See all available scope names
list(SCOPES.keys())

In [ ]:
# Look up a specific scope
SCOPES["franklin"]

In [ ]:
# The 15-county MORPC region covers multiple counties — for_param holds their FIPS codes
SCOPES["region15"]

## 2. Scales — choosing resolution

A scale sets the resolution of the data. `valid_scale()` confirms a scale name is recognized and returns its Census summary level code.

In [ ]:
# Validate a scale and see its summary level code
valid_scale("county")     # sumlevel 050

In [ ]:
valid_scale("tract")      # sumlevel 140

In [ ]:
# Unrecognized names raise a ValueError listing the available options
try:
    valid_scale("neighborhood")
except ValueError as e:
    print(e)

## 3. Fetching geometries

`fetch_geos_from_scale_scope(scope, scale)` is the primary function for downloading spatial data. Pass a scope name and an optional scale name — it returns a GeoDataFrame.

> **Network required** — the cells below make live calls to the Census API and TIGERweb REST API.

In [ ]:
# Get county boundaries for the 15-county MORPC region
# Omitting scale returns geographies at the scope's own level (counties)
region_counties = fetch_geos_from_scale_scope(scope="region15")
region_counties

In [ ]:
region_counties.plot()

In [ ]:
# Get all census tracts within Franklin County
franklin_tracts = fetch_geos_from_scale_scope(scope="franklin", scale="tract")
franklin_tracts.plot()

## 4. Working with GEOIDFQs

Every geography returned by the Census API has a **GEOIDFQ** (fully-qualified geographic identifier) — a string that encodes the geography type and its component codes.

For example, `1400000US39041010100` is the GEOIDFQ for census tract 101 in Delaware County, Ohio:

```
140  00  00  US  39    041     010100
─┬─  ─┬  ─┬  ─┬  ─┬─  ─┬─    ──┬──
 │    │   │   │   │    │       tract
 │    │   │   │   │    county
 │    │   │   │   state
 │    │   │   literal
 │    │   geocomp
 │    variant
 sumlevel (140 = census tract)
```

`GeoIDFQ` lets you parse, inspect, and construct these strings.

In [ ]:
# Parse a GEOIDFQ returned by the Census API
tract = GeoIDFQ.parse("1400000US39041010100")
tract

In [ ]:
# .parts gives the named geographic components
tract.parts

In [ ]:
# .geoid is the short-form ID used in REST API queries (the part after 'US')
tract.geoid

In [ ]:
# Build a GEOIDFQ from component codes
county = GeoIDFQ.build("050", {"state": "39", "county": "049"})
str(county)    # Franklin County, Ohio

In [ ]:
# For geographies with variant codes (e.g. Congressional districts), pass variant explicitly
# Variant = Congress number minus 100, so the 119th Congress = "19"
cd = GeoIDFQ.build("500", {"state": "39", "cd": "12"}, variant="19")
str(cd)    # Ohio's 12th Congressional District, 119th Congress

In [ ]:
# GEOIDFQs from a fetch can be parsed directly from the GEO_ID column
# (network required — uses region_counties from section 3)
geoidfqs = region_counties["GEO_ID"].tolist()
parsed = [GeoIDFQ.parse(g) for g in geoidfqs]

# Show the state and county FIPS codes for each geography
[(g.parts["state"], g.parts["county"]) for g in parsed]